In [ ]:
#Langchain
from langchain.tools import Tool
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain.prompts import PromptTemplate
from langchain_openai import ChatOpenAI
from langchain.chains import LLMChain
from langchain.agents import initialize_agent, AgentType
from langchain_community.tools.tavily_search import TavilySearchResults

In [ ]:
import requests
from bs4 import BeautifulSoup

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv("/Users/nareshchaurasia/nc/PYTHON-ARCHITECT/Python-Immersive-AI-MAC/.env_rag")

api_key = os.getenv("CO_API_KEY")
print(api_key)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
#prompt templates
prompt_template = PromptTemplate(
    input_variables=["name"],
    template="Hello, {name}! How can I help you today?"
)

formatted_prompt = prompt_template.format(name="David")
print(formatted_prompt)

In [ ]:
chat_model = ChatOpenAI(model="gpt-4o-mini")

response = chat_model.invoke("What is Capital of USA?")
print(response.content)

In [ ]:
#LLM Chains
llm = ChatOpenAI(model="gpt-4o-mini")

learn_template = """
I want you to act as a consultant for a AI training
Return a list of topics and why it is important to learn in given area of AI
The description should be relevant to recent advancement in AI
What are some good topics to learn in {AI_topic}
"""

learn_prompt = PromptTemplate(
    input_variables=["AI_topic"],
    template=learn_template,
)

description = "Deep learning"

chain = LLMChain(llm=llm, prompt=learn_prompt)

result = chain.invoke({"AI_topic": description})
print(result["text"])

### Using Simple Function

In [ ]:
# Import the required classes
from langchain.agents import initialize_agent, AgentType
from langchain.chat_models import ChatOpenAI
from langchain.tools import Tool

# Define a custom tool function
def my_tool_function(query: str) -> str:
    # Simply returns the input query with a prefix
    return f"Tool1 response: {query}"

# Define a custom tool function
def my_tool_function2(query: str) -> str:
    # Simply returns the input query with a prefix
    return f"Tool response: {query}"

# Convert the Python function into a LangChain Tool
my_tool = Tool.from_function(
    func=my_tool_function,         # Function to execute
    name="simple_tool",            # Tool name used by the agent
    description="A simple tool"    # Description helps the agent decide when to use it
)

# Convert the Python function into a LangChain Tool
my_tool2 = Tool.from_function(
    func=my_tool_function2,         # Function to execute
    name="simple_tool2",            # Tool name used by the agent
    description="A simple tool2"    # Description helps the agent decide when to use it
)

# Create a Tavily search tool (requires TAVILY_API_KEY)
# tavily_search = TavilySearchResults(max_results=2)

# Initialize the language model
llm = ChatOpenAI(model="gpt-4o-mini")

# List of tools available to the agent
# tools = [tavily_search, my_tool]   # Agent can use both Tavily Search and custom tool
tools = [my_tool, my_tool2]                    # Agent can use only the custom tools

# Create the agent
agent = initialize_agent(
    tools,                                       # Tools available to the agent
    llm,                                         # Language model
    agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,  # Uses ReAct reasoning based on tool descriptions
    verbose=True                                 # Shows the agent's reasoning steps
)

# Ask the agent a question
response = agent.run("What's the weather like today in London?")

# Print the final response
print(response)

Certainly. This output shows how a **ReAct (Reason + Act)** agent thinks step-by-step while trying to answer your question.

Let's go through each part.

**1. Agent starts**

```text
> Entering new AgentExecutor chain...
```

This indicates that the LangChain agent has started executing your request.

---

**2. First Thought**

```text
I need to find out the current weather conditions in London.
I will use the simple_tool to query the weather information.
```

This is the agent's reasoning.

It analyzes the question:

> "What's the weather like today in London?"

Since the only available tool is:

```python
tools = [my_tool]
```

and its description is:

```python
description="A simple tool"
```

the agent decides that this is the only tool it can use.

---

**3. Action**

```text
Action: simple_tool
```

The agent has decided to call your custom tool.

Internally, it is equivalent to:

```python
my_tool_function("current weather in London")
```

---

**4. Action Input**

```text
Action Input:
"current weather in London"
```

This is the string passed to your tool.

So your function receives:

```python
query = "current weather in London"
```

---

**5. Observation**

```text
Observation:
Tool response: current weather in London
```

This is the output returned by your tool.

Remember your function:

```python
def my_tool_function(query):
    return f"Tool response: {query}"
```

So it simply echoes the input.

It **does not**:

- Search Google
- Call a weather API
- Know London's temperature

It only returns:

```text
Tool response: current weather in London
```

---

**6. Second Thought**

```text
The tool provided a general response,
but I need specific weather details...
```

The agent reasons:

> "That wasn't useful."

Because it expected weather information but only got an echoed sentence.

---

**7. Another Action**

The agent tries again.

```text
Action:
simple_tool

Action Input:
What's the current temperature and weather conditions in London?
```

Again your function receives:

```python
my_tool_function(
    "What's the current temperature and weather conditions in London?"
)
```

Output:

```text
Tool response:
What's the current temperature and weather conditions in London?
```

Again, no real weather information.

---

**8. Loop Continues**

The agent keeps thinking:

```text
Maybe I should ask differently.
```

It tries different queries such as:

```text
London weather details today...
```

Then:

```text
Provide detailed weather information...
```

Then:

```text
Get current weather report...
```

Each time your function simply echoes the input instead of returning actual weather data.

---

**9. Final Thought**

Eventually the agent realizes:

```text
The tool is not providing specific weather details.
```

The ReAct agent concludes:

> "This tool cannot answer my question."

---

**10. Final Answer**

```text
I am unable to provide the current weather information...
```

The agent gives up because it has exhausted reasonable attempts.

---

**Why did it keep trying?**

Because **ReAct agents are designed to think and retry**.

The execution cycle is:

```text
Question
     │
     ▼
Thought
     │
     ▼
Choose Tool
     │
     ▼
Execute Tool
     │
     ▼
Observe Result
     │
     ▼
Enough?
 ┌───────┴────────┐
 │                │
No               Yes
 │                │
 ▼                ▼
Think Again     Final Answer
```

This retry behavior is one of the key features of the ReAct agent.

---

**Why couldn't it answer?**

Your tool is intentionally very simple:

```python
def my_tool_function(query):
    return f"Tool response: {query}"
```

No matter what is asked:

```text
"Weather?"
```

or

```text
"Temperature?"
```

or

```text
"Humidity?"
```

it only returns:

```text
Tool response: <input>
```

There is **no external data source**, so the tool cannot provide real-time weather information.

### Using DuckDuckGoSearchRun

DuckDuckGoSearchRun is a LangChain tool that lets an AI agent search the web using the DuckDuckGo search engine.

In [18]:
from langchain_community.tools import DuckDuckGoSearchRun

dg_search = DuckDuckGoSearchRun()

tools = [dg_search, my_tool]                    # Agent can use both tools

# Create the agent
agent = initialize_agent(
    tools,                                       # Tools available to the agent
    llm,                                         # Language model
    agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,  # Uses ReAct reasoning based on tool descriptions
    verbose=True                                 # Shows the agent's reasoning steps
)

# Ask the agent a question
response = agent.run("What's the weather like today in London?")

# Print the final response
print(response)



> Entering new AgentExecutor chain...
I need to find the current weather information for London. 
Action: duckduckgo_search 
Action Input: "current weather London" 

/Users/nareshchaurasia/nc/PYTHON-ARCHITECT/Python-Immersive-AI-MAC/ALL-VENV/.venv_aman_agents_1/lib/python3.10/site-packages/langchain_community/utilities/duckduckgo_search.py:63: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:



Observation: No good DuckDuckGo Search Result was found
Thought:

/Users/nareshchaurasia/nc/PYTHON-ARCHITECT/Python-Immersive-AI-MAC/ALL-VENV/.venv_aman_agents_1/lib/python3.10/site-packages/langchain_community/chat_models/openai.py:495: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  response = response.dict()
/Users/nareshchaurasia/nc/PYTHON-ARCHITECT/Python-Immersive-AI-MAC/ALL-VENV/.venv_aman_agents_1/lib/python3.10/site-packages/langchain_community/utilities/duckduckgo_search.py:63: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


It seems that the search did not yield useful results. I will try again with a more specific query to find the current weather in London. 
Action: duckduckgo_search 
Action Input: "London weather today" 
Observation: No good DuckDuckGo Search Result was found
Thought:

...

/Users/nareshchaurasia/nc/PYTHON-ARCHITECT/Python-Immersive-AI-MAC/ALL-VENV/.venv_aman_agents_1/lib/python3.10/site-packages/langchain_community/chat_models/openai.py:495: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  response = response.dict()
/Users/nareshchaurasia/nc/PYTHON-ARCHITECT/Python-Immersive-AI-MAC/ALL-VENV/.venv_aman_agents_1/lib/python3.10/site-packages/langchain_community/utilities/duckduckgo_search.py:63: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


I need to find specific weather details such as temperature, conditions, and forecasts for London. Since the simple tool did not yield detailed information, I will try to look up current weather conditions again using a different search query to obtain the needed details.  
Action: duckduckgo_search  
Action Input: "London weather report"  
Observation: The London Mayor is distinguished from the Lord Mayor of London, who heads the City of London … Discover your ultimate guide to London. From the best activities in the city to top restaurants, bars … Mar 16, 2026 · As the official guide to London, our team of experts bring together the very best of … 3 days ago · London, city, capital of the United Kingdom. It is among the oldest of the world’s great … Royalty-spotting and museum trips are top to-do’s in London—from iconic landmarks like …
Thought:

/Users/nareshchaurasia/nc/PYTHON-ARCHITECT/Python-Immersive-AI-MAC/ALL-VENV/.venv_aman_agents_1/lib/python3.10/site-packages/langchain_community/chat_models/openai.py:495: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  response = response.dict()
/Users/nareshchaurasia/nc/PYTHON-ARCHITECT/Python-Immersive-AI-MAC/ALL-VENV/.venv_aman_agents_1/lib/python3.10/site-packages/langchain_community/utilities/duckduckgo_search.py:63: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


It appears that I am still unable to retrieve specific and current weather information for London. I will attempt one more time using a different search phrase to see if I can get the necessary details.  
Action: duckduckgo_search  
Action Input: "current weather in London today"  
Observation: No good DuckDuckGo Search Result was found
Thought:I have exhausted my attempts to find current weather information through both search tools without success. It seems I need to gather more comprehensive data or try a different approach to find the weather in London. Since I have not been able to get the information I need, I will conclude that I currently cannot provide the answer to the question about the weather in London today. 

Final Answer: I am unable to provide the current weather information for London today.

> Finished chain.
I am unable to provide the current weather information for London today.


/Users/nareshchaurasia/nc/PYTHON-ARCHITECT/Python-Immersive-AI-MAC/ALL-VENV/.venv_aman_agents_1/lib/python3.10/site-packages/langchain_community/chat_models/openai.py:495: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  response = response.dict()


In [ ]:
prompt_template = "Summarize the following content: {content}"
llm = ChatOpenAI(model="gpt-4o-mini")

llm_chain = LLMChain(
    llm=llm,
    prompt=PromptTemplate.from_template(prompt_template)
)

summarize_tool = Tool.from_function(
    func=llm_chain.run,
    name="Summarizer",
    description="Summarizes a web page"
)

In [ ]:
tools = [dg_search, summarize_tool]

agent = initialize_agent(
    tools=tools,
    agent_type=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    llm=llm,
    verbose=True,handle_parsing_errors=True
)

In [ ]:
response = agent.invoke({"input": "Who invented the World Wide Web and what impact did it have?"})
print(response["output"])